  # 资源分配问题

 ## 包导入

In [44]:
from numpy import cumsum
from time import time
import random as rd

  ## 类定义

In [45]:
class node():  # 节点类
    def __init__(self, name=None, resourse=None):  # 初始化节点
        self.name = name  # 名称
        self.resourse = resourse  # 资源
        self.nets = set()  # 网络集合

  ## 函数定义
  ### 读取文件信息


In [46]:
def readNodeInfo():  # 读取节点信息
    with open("design.are") as file:  # 打开节点文件
        nodes = {}  # 创建节点字典
        for line in file:  # 按行遍历文件
            name = line.split()[0]  # 取当前行首字符为节点名
            nodes[name] = node(name)  # 新建节点键值到字典
    return nodes  # 返回字典

In [47]:
def readNetInfo():  # 读取网络信息
    names = []  # 同网络节点名称列表
    with open("design.net") as file:
        for line in file:
            if 's' in line.split():  # 当前行中包含网络首节点s
                for name in names:  # 将之前节点映射到前首节点网络
                    nodes[name].nets.add(names[0])
                names.clear()  # 清空已映射节点
            names.append(line.split()[0])  # 添加当前行的节点
        nodes[name].nets.add(names[0])  # 将剩余节点映射到当前网络

  ### 定义分配函数

In [48]:
def allocation():  # 分配单个染色体
    return [rd.randint(0, NF-1) for _ in nodes]  # 返回映射所有节点fpga号的随机染色体编码

  ### 定义结果函数

In [49]:
def writeResult():  # 输出分配结果
    buffer = ''  # 临时字符串
    for fpga in range(NF):  # 遍历4个fpga
        buffer += 'F'+str(fpga)+' '  # 添加fpga编号
        for index, node in enumerate(nodes.values()):  # 遍历所有节点
            if best[index] == fpga:  # 若节点通过最佳染色体映射为当前fpga号
                buffer += node.name+' '  # 添加当前节点名称
        buffer += '\n'
    print(buffer)  # 打印最优fpga节点分配结果
    buffer += str(links[index1])  # 添加最优板间连线结果
    with open("./result.res", 'w') as file:  # 打开结果文件
        file.write(buffer)  # 写入最优结果

  ### 定义资源差异函数

In [50]:
def resourseCal():
    return 0

  ### 定义板间连线函数

In [51]:
def linkCal(ind):  # 计算FPGA板间连线总和
    link = 0  # 初始化连线数量
    fpgaNets = [set() for _ in range(NF)]  # 4个fpga的网络集合列表
    for index, node in enumerate(nodes.values()):  # 遍历所有节点
        fpgaNets[ind[index]] |= node.nets  # 将节点网络添加到通过染色体映射的fpga网络
    for index, net1 in enumerate(fpgaNets):  # 遍历所有fpga网络
        for net2 in fpgaNets[index+1:]:  # 遍历之后的fpga网络
            link += len(net1 & net2)  # 两板相同网络数即为板间连线数
    return link

 ### 定义适应值函数

In [52]:
def fitCal(index, link):  # 计算link对应的适应值
    if len(set(pop[index])) == NF:  # 若有节点的fpga板数量为4
        return 1/link  # 返回适应值
    else:  # 否则说明存在fpga未分配节点
        return 0  # 适应值为0

 ### 定义筛选函数

In [53]:
def screenPop():  # 筛选种群个体
    sumFit = cumsum(fitness)  # 计算累加适应值列表（生成轮盘）
    for i, ind1 in enumerate(pop):  # 遍历每个个体
        for j, ind2 in enumerate(pop):  # 遍历新的个体（轮盘转圈）
            if rd.random() < sumFit[j] and fitness[i] < fitness[j]:  # 若轮盘选中个体适应值较大
                ind1 = ind2  # 以新个体淘汰当前个体
                break  # 轮盘停止转动

 ### 定义交叉函数

In [54]:
def crossPop():  # 种群个体交配
    for i in range(int(NP/2)):  # 遍历雄性个体
        j = rd.randint(int(NP/2), NP-1)  # 寻找配偶
        if rd.random() < PC:  # 求偶成功
            index = rd.randint(1, len(nodes))  # 染色体交叉点
            child1 = pop[i][:index]+pop[j][index:]  # 子染色体
            child2 = pop[j][:index]+pop[i][index:]  # 或子代个体
            if len(set(child1) & set(child1)) == NF:  # 若子染色体包含所有fpga
                pop[i], pop[j] = child1, child2  # 牺牲父代，保留子代

 ### 定义变异函数

In [55]:
def mutatePop():  # 种群个体变异
    for ind in pop:
        if rd.random() < PW:  # 触发变异
            # gene1, gene2 = rd.sample(ind, 2)  # 浅拷贝，无法修改原值
            # gene1, gene2 = gene2, gene1
            # i, j = rd.sample(range(len(ind)), 2)
            # ind[i], ind[j] = ind[j], ind[i]  # 索引，可以修改原值
            rd.shuffle(ind)  # 超级变异

  ## 执行代码

 ### 设定参数

In [56]:
NF = 4  # FPGA数量
MI = 10**3  # 最大迭代次数
PC = 0.8  # 交叉概率
PW = 0.2  # 变异概率
NP = 100  # 染色体规模

 ### 遗传算法

In [57]:
nodes = readNodeInfo()  # 读取节点文件
readNetInfo()  # 读取线网文件
pop = [allocation() for _ in range(NP)]  # 初始化N个体种群
tstart = time()  # 开始计时
for iter in range(MI):  # 种群迭代
    links = [linkCal(ind) for ind in pop]  # 板间连线总和列表
    fitness = [fitCal(i, link) for i, link in enumerate(links)]  # 个体适应值列表
    maxFit, minFit = max(fitness), min(fitness)  # 最大、最小适应值
    index1 = fitness.index(maxFit)  # 最大适应值索引
    index2 = fitness.index(minFit)  # 最小适应值索引
    minLink = links[index1]  # 最小连线总和，不使用min(links)是因为fpga不一定都有节点
    best = pop[index1]  # 最佳适应性个体
    screenPop()  # 筛选种群个体
    crossPop()  # 种群个体交配
    mutatePop()  # 种群个体变异
    pop[index2] = best  # 淘汰最差并保留最佳适应性个体
    if not iter % (MI/10):  # 迭代进度达到设置的10%
        print('迭代次数：%3d\t时间：%.2fs\t板间连线：%2d' %
              (iter+1, time()-tstart, minLink))
print('最小连线：%3d\t适应值：%.1f\t分配关系：' % (minLink, maxFit))
writeResult() # 打印结果


迭代次数：  1	时间：0.01s	板间连线：61
迭代次数：101	时间：0.51s	板间连线：42
迭代次数：201	时间：1.00s	板间连线：30
迭代次数：301	时间：1.43s	板间连线：13
迭代次数：401	时间：1.80s	板间连线： 4
迭代次数：501	时间：2.15s	板间连线： 3
迭代次数：601	时间：2.49s	板间连线： 5
迭代次数：701	时间：2.84s	板间连线： 3
迭代次数：801	时间：3.21s	板间连线： 4
迭代次数：901	时间：3.54s	板间连线： 3
最小连线：  3	适应值：0.3	分配关系：
F0 g0 g1 g2 g3 g4 g5 g6 g7 g8 g9 g10 g11 g12 g13 g14 g15 g16 g17 g18 g19 g20 g21 g22 g23 g24 g25 g26 g27 g28 g29 gp0 gp1 gp2 gp3 gp4 gp5 gp6 gp7 gp8 gp9 gp10 gp11 gp13 gp14 gp16 gp18 gp19 gp20 gp21 
F1 g30 gp17 
F2 gp12 
F3 gp15 

